In [0]:
dbutils.secrets.listScopes()

[SecretScope(name='databrickscope')]

In [0]:
from pyspark.sql.functions import sum, col, when

In [0]:
%sql
CREATE EXTERNAL LOCATION transformed_location
URL 'abfss://transformed@mistoragelago.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credencialdestino);

In [0]:
display(
    dbutils.fs.ls(
        "abfss://rawdata@mistoragelago.dfs.core.windows.net/"
    )
)

path,name,size,modificationTime
abfss://rawdata@mistoragelago.dfs.core.windows.net/accounts.csv,accounts.csv,4670,1780075518000
abfss://rawdata@mistoragelago.dfs.core.windows.net/data_dictionary.csv,data_dictionary.csv,996,1780075498000
abfss://rawdata@mistoragelago.dfs.core.windows.net/products.csv,products.csv,171,1780075587000
abfss://rawdata@mistoragelago.dfs.core.windows.net/sales_pipeline.csv,sales_pipeline.csv,637773,1780075567000
abfss://rawdata@mistoragelago.dfs.core.windows.net/sales_teams.csv,sales_teams.csv,1284,1780075543000


In [0]:
display(
    dbutils.fs.ls(
        "abfss://transformed@mistoragelago.dfs.core.windows.net/"
    )
)

[]

In [0]:
accounts_df=spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("abfss://rawdata@mistoragelago.dfs.core.windows.net/accounts.csv")
display(accounts_df)


account,sector,year_established,revenue,employees,office_location,subsidiary_of
Acme Corporation,technolgy,1996,1100.04,2822,United States,null
Betasoloin,medical,1999,251.41,495,United States,null
Betatech,medical,1986,647.18,1185,Kenya,null
Bioholding,medical,2012,587.34,1356,Philipines,null
Bioplex,medical,1991,326.82,1016,United States,null
Blackzim,retail,2009,497.11,1588,United States,null
Bluth Company,technolgy,1993,1242.32,3027,United States,Acme Corporation
Bubba Gump,software,2002,987.39,2253,United States,null
Cancity,retail,2001,718.62,2448,United States,null
Cheers,entertainment,1993,4269.9,6472,United States,Massive Dynamic


In [0]:
data_dictionary_df=spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("abfss://rawdata@mistoragelago.dfs.core.windows.net/data_dictionary.csv")
products_df=spark.read.format("csv").option("header", "true").option("inferSchema","true").load("abfss://rawdata@mistoragelago.dfs.core.windows.net/products.csv")
sales_pipeline_df=spark.read.format("csv").option("header", "true").option("inferSchema","true").load("abfss://rawdata@mistoragelago.dfs.core.windows.net/sales_pipeline.csv")
sales_teams_df=spark.read.format("csv").option("header", "true").option("inferSchema","true").load("abfss://rawdata@mistoragelago.dfs.core.windows.net/sales_teams.csv")




In [0]:
print(accounts_df.columns)
print(data_dictionary_df.columns)
print(products_df.columns)
print(sales_pipeline_df.columns)
print(sales_teams_df.columns)

['account', 'sector', 'year_established', 'revenue', 'employees', 'office_location', 'subsidiary_of']
['Table', 'Field', 'Description']
['product', 'series', 'sales_price']
['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage', 'engage_date', 'close_date', 'close_value']
['sales_agent', 'manager', 'regional_office']


In [0]:
accounts_df=accounts_df.withColumnRenamed("subsidiary_of","parent_company")
data_dictionary_df=data_dictionary_df.withColumnRenamed("Table","tabla")
data_dictionary_df=data_dictionary_df.withColumnRenamed("Field","field")
data_dictionary_df=data_dictionary_df.withColumnRenamed("Description","description")
accounts_df.show()

+----------------+-------------+----------------+-------+---------+---------------+----------------+
|         account|       sector|year_established|revenue|employees|office_location|  parent_company|
+----------------+-------------+----------------+-------+---------+---------------+----------------+
|Acme Corporation|    technolgy|            1996|1100.04|     2822|  United States|            NULL|
|      Betasoloin|      medical|            1999| 251.41|      495|  United States|            NULL|
|        Betatech|      medical|            1986| 647.18|     1185|          Kenya|            NULL|
|      Bioholding|      medical|            2012| 587.34|     1356|     Philipines|            NULL|
|         Bioplex|      medical|            1991| 326.82|     1016|  United States|            NULL|
|        Blackzim|       retail|            2009| 497.11|     1588|  United States|            NULL|
|   Bluth Company|    technolgy|            1993|1242.32|     3027|  United States|Acme Cor

In [0]:
null_accounts_df=accounts_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in accounts_df.columns])
null_accounts_df.show()


+-------+------+----------------+-------+---------+---------------+--------------+
|account|sector|year_established|revenue|employees|office_location|parent_company|
+-------+------+----------------+-------+---------+---------------+--------------+
|      0|     0|               0|      0|        0|              0|            70|
+-------+------+----------------+-------+---------+---------------+--------------+



In [0]:
null_counts_data_dictionary=data_dictionary_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in data_dictionary_df.columns])
null_counts_products_df=products_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in products_df.columns])
null_counts_sales_pipeline_df=sales_pipeline_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in sales_pipeline_df.columns])
null_counts_sales_teams_df=sales_teams_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in sales_teams_df.columns])


In [0]:
null_counts_products_df.show()

+-------+------+-----------+
|product|series|sales_price|
+-------+------+-----------+
|      0|     0|          0|
+-------+------+-----------+



In [0]:
null_counts_sales_pipeline_df.show()

+--------------+-----------+-------+-------+----------+-----------+----------+-----------+
|opportunity_id|sales_agent|product|account|deal_stage|engage_date|close_date|close_value|
+--------------+-----------+-------+-------+----------+-----------+----------+-----------+
|             0|          0|      0|   1425|         0|        500|      2089|       2089|
+--------------+-----------+-------+-------+----------+-----------+----------+-----------+



In [0]:
accounts_df=accounts_df.fillna({"parent_company":"independent"})
sales_pipeline_df=sales_pipeline_df.fillna({"account":"unknown"})


In [0]:
null_accounts_df=accounts_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in accounts_df.columns])
null_accounts_df.show()

+-------+------+----------------+-------+---------+---------------+--------------+
|account|sector|year_established|revenue|employees|office_location|parent_company|
+-------+------+----------------+-------+---------+---------------+--------------+
|      0|     0|               0|      0|        0|              0|             0|
+-------+------+----------------+-------+---------+---------------+--------------+



## after renaming columns and filling the null values for some of the datasets, we are going to move the data in the Lake Gen 2 container rawdata to container "transformed"

In [0]:
accounts_df.write.option("header",True).csv("abfss://transformed@mistoragelago.dfs.core.windows.net/accounts.csv")


In [0]:
data_dictionary_df.write.option("header",True).csv("abfss://transformed@mistoragelago.dfs.core.windows.net/data_dictionary.csv")
products_df.write.option("header",True).csv("abfss://transformed@mistoragelago.dfs.core.windows.net/products.csv")
sales_pipeline_df.write.option("header",True).csv("abfss://transformed@mistoragelago.dfs.core.windows.net/sales_pipeline.csv")
sales_teams_df.write.option("header",True).csv("abfss://transformed@mistoragelago.dfs.core.windows.net/sales_teams.csv")